# Exp6 Router Ablation — Train Qwen2.5-0.5B-Instruct

**Size**: qwen25_05b  
**Base model**: `unsloth/Qwen2.5-0.5B-Instruct`  
**Output repo**: `daredevil467/hanoi-router-qwen25-05b-v6`  
**Data**: [`daredevil467/hanoi-weather-router-data`](https://huggingface.co/datasets/daredevil467/hanoi-weather-router-data) (train 3471, val 385)  
**Effective batch**: 16 × 2 = 32

Run on Colab A100 (premium GPU). Before running, set `HF_TOKEN` in Colab Secrets (🔑 panel).

In [8]:
%%capture
!pip install -U "unsloth[colab-new]" "trl>=0.17,<0.19" "huggingface_hub>=0.26" datasets


In [9]:
import os, torch, subprocess
from google.colab import userdata
from huggingface_hub import login

# HF login from Colab Secrets (🔑 icon in left panel)
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN in Colab Secrets before running'
login(token=HF_TOKEN)

gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode().strip()
print(f'GPU: {gpu}')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported(), 'A100 (or better) required for bf16 training'


GPU: NVIDIA A100-SXM4-80GB, 81920 MiB
bf16 supported: True


In [10]:
# ═══════════════════════ CONFIG (Exp6 shared recipe) ═══════════════════════
BASE_MODEL     = 'unsloth/Qwen2.5-0.5B-Instruct'
HF_REPO        = 'daredevil467/hanoi-router-qwen25-05b-v6'
HF_DATA_REPO   = 'daredevil467/hanoi-weather-router-data'

MAX_SEQ_LENGTH = 1024
LORA_R         = 32          # Identical to v4 recipe
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.1
EPOCHS         = 10
BATCH_SIZE     = 16
GRAD_ACCUM     = 2
LR             = 2e-4
WARMUP_RATIO   = 0.06
EVAL_STEPS     = 50
OUTPUT_DIR     = '/content/outputs'

print(f'Model      : {BASE_MODEL}')
print(f'LoRA       : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'Batch      : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective')
print(f'Epochs     : {EPOCHS}')
print(f'LR         : {LR}')
print(f'HF target  : {HF_REPO}')


Model      : unsloth/Qwen2.5-0.5B-Instruct
LoRA       : r=32, alpha=64, dropout=0.1
Batch      : 16 x 2 = 32 effective
Epochs     : 10
LR         : 0.0002
HF target  : daredevil467/hanoi-router-qwen25-05b-v6


In [11]:
# ═══════════════════════ DATA DOWNLOAD from HF ═══════════════════════
from huggingface_hub import hf_hub_download
import json as _json

train_file = hf_hub_download(repo_id=HF_DATA_REPO, filename='multitask_train_v6_clean.jsonl', repo_type='dataset')
val_file   = hf_hub_download(repo_id=HF_DATA_REPO, filename='multitask_val_v6_clean.jsonl',   repo_type='dataset')
prompt_file= hf_hub_download(repo_id=HF_DATA_REPO, filename='system_prompt.txt',              repo_type='dataset')

with open(train_file, encoding='utf-8') as f:
    train_raw = [_json.loads(l) for l in f]
with open(val_file, encoding='utf-8') as f:
    val_raw = [_json.loads(l) for l in f]
with open(prompt_file, encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()

print(f'Train: {len(train_raw)} samples')
print(f'Val:   {len(val_raw)} samples')
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
assert len(train_raw) == 3471, f'Expected 3471 train rows, got {len(train_raw)}'
assert len(val_raw)   == 385,  f'Expected 385 val rows, got {len(val_raw)}'


Train: 3471 samples
Val:   385 samples
System prompt: 1761 chars


In [12]:
# ═══════════════════════ LOAD BASE MODEL ═══════════════════════
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = False,
)
print(f'Model loaded: {BASE_MODEL}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen2.5-0.5B-Instruct
Params: 494,032,768


In [13]:
# ═══════════════════════ APPLY LoRA ═══════════════════════
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable: 17,596,416 / 511,629,184 (3.44%)


In [14]:
# ═══════════════════════ TOKENIZER SETUP (manual ChatML) ═══════════════════════
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'right'

# Verify ChatML special tokens exist (Qwen2/3 family)
im_start_ids = tokenizer.encode('<|im_start|>', add_special_tokens=False)
im_end_ids   = tokenizer.encode('<|im_end|>',   add_special_tokens=False)
print(f'<|im_start|> token ids: {im_start_ids}')
print(f'<|im_end|> token ids:   {im_end_ids}')
assert len(im_start_ids) == 1 and len(im_end_ids) == 1, 'ChatML special tokens missing'


<|im_start|> token ids: [151644]
<|im_end|> token ids:   [151645]


In [15]:
# ═══════════════════════ FORMAT RECORDS (ChatML, identical to v4) ═══════════════════════
import json
from datasets import Dataset

IM_START = '<|im_start|>'
IM_END   = '<|im_end|>'
NL       = chr(10)

def format_record(rec):
    user_msg = str(rec.get('input', '')).strip()
    ctx = rec.get('context')
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(',', ':'))
        user_msg = '[CONTEXT: ' + ctx_str + ']' + NL + user_msg
    out = rec.get('output', {})
    if isinstance(out, str):
        out = json.loads(out)
    output_dict = {
        'intent':     out['intent'],
        'scope':      out['scope'],
        'confidence': round(float(out.get('confidence', 0.9)), 2),
    }
    rw = out.get('rewritten_query')
    if rw and str(rw).strip():
        output_dict['rewritten_query'] = str(rw).strip()
    text  = IM_START + 'system'    + NL + SYSTEM_PROMPT + IM_END + NL
    text += IM_START + 'user'      + NL + user_msg      + IM_END + NL
    text += IM_START + 'assistant' + NL + json.dumps(output_dict, ensure_ascii=False) + IM_END + NL
    return text

train_texts = [format_record(r) for r in train_raw]
val_texts   = [format_record(r) for r in val_raw]

lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print(f'Avg tokens: {sum(lengths)/len(lengths):.0f}, Max: {max(lengths)}, Over {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}')

raw_train = Dataset.from_dict({'text': train_texts})
raw_val   = Dataset.from_dict({'text': val_texts})
print(f'Train: {len(raw_train)}, Val: {len(raw_val)}')


Avg tokens: 639, Max: 674, Over 1024: 0
Train: 3471, Val: 385


In [16]:
# ═══════════════════════ COMPLETION-ONLY MASKING (v3 key fix) ═══════════════════════
# Mask everything up to '<|im_start|>assistant\n'. Loss only on the JSON response.
ASSISTANT_MARKER = '<|im_start|>assistant' + chr(10)
assistant_ids = tokenizer.encode(ASSISTANT_MARKER, add_special_tokens=False)
print(f'Assistant marker ids: {assistant_ids}')

def tokenize_with_masking(examples):
    enc = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_tensors=None,
    )
    all_labels = []
    for input_ids in enc['input_ids']:
        labels = [-100] * len(input_ids)
        marker_len = len(assistant_ids)
        for i in range(len(input_ids) - marker_len + 1):
            if input_ids[i:i+marker_len] == assistant_ids:
                start = i + marker_len
                for j in range(start, len(input_ids)):
                    labels[j] = input_ids[j]
                break
        else:
            labels = list(input_ids)  # fallback (should not happen)
        all_labels.append(labels)
    enc['labels'] = all_labels
    return enc

train_dataset = raw_train.map(tokenize_with_masking, batched=True, remove_columns=['text'])
val_dataset   = raw_val.map(tokenize_with_masking, batched=True, remove_columns=['text'])

# Sanity check
sl = train_dataset[0]['labels']
masked = sum(1 for x in sl if x == -100)
print(f'Sample: {len(sl)} total, {masked} masked ({masked/len(sl)*100:.1f}%), {len(sl)-masked} active')


Assistant marker ids: [151644, 77091, 198]


Map:   0%|          | 0/3471 [00:00<?, ? examples/s]

Map:   0%|          | 0/385 [00:00<?, ? examples/s]

Sample: 639 total, 615 masked (96.2%), 24 active


In [17]:
# ═══════════════════════ TRAINER SETUP ═══════════════════════
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = 'cosine',
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = EVAL_STEPS,
    save_strategy               = 'steps',
    save_steps                  = EVAL_STEPS,
    save_total_limit            = 2,
    metric_for_best_model       = 'eval_loss',
    load_best_model_at_end      = True,
    bf16                        = True,
    fp16                        = False,
    optim                       = 'adamw_torch_fused',
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    max_seq_length              = MAX_SEQ_LENGTH,
    report_to                   = 'none',
)

early_stop = EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.001)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    data_collator    = data_collator,
    packing          = False,
    callbacks        = [early_stop],
)
print('Trainer ready')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready


In [18]:
# ═══════════════════════ TRAIN ═══════════════════════
trainer_stats = trainer.train()
print(trainer_stats)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,471 | Num Epochs = 10 | Total steps = 1,090
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 17,596,416 of 511,629,184 (3.44% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.213344,0.215816
100,0.176742,0.172445
150,0.120421,0.146118
200,0.120266,0.140231
250,0.096110,0.135073
300,0.110310,0.133340
350,0.084585,0.127757
400,0.077308,0.127078
450,0.071185,0.128217
500,0.070423,0.124592


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=800, training_loss=0.10469022523611785, metrics={'train_runtime': 603.4229, 'train_samples_per_second': 57.522, 'train_steps_per_second': 1.806, 'total_flos': 3.862993379838566e+16, 'train_loss': 0.10469022523611785})


In [19]:
# ═══════════════════════ SAVE ADAPTER + MERGE + PUSH to HF ═══════════════════════
# 1. Save LoRA adapter locally
adapter_dir = '/content/outputs/lora_adapter'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Adapter saved to {adapter_dir}')

# 2. Free disk (checkpoints) before merge
import shutil, gc
from pathlib import Path
out = Path(OUTPUT_DIR)
for ckpt in sorted(out.glob('checkpoint-*')):
    print(f'Deleting {ckpt.name}...')
    shutil.rmtree(ckpt)
torch.cuda.empty_cache()
gc.collect()

# 3. Push merged model (16-bit) to HF Hub — used by eval notebook 06
print(f'Pushing merged model to HF: {HF_REPO}')
model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method='merged_16bit',
    token=HF_TOKEN,
)
print(f'Pushed: https://huggingface.co/{HF_REPO}')


Adapter saved to /content/outputs/lora_adapter
Deleting checkpoint-550...
Deleting checkpoint-800...
Pushing merged model to HF: daredevil467/hanoi-router-qwen25-05b-v6


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n25-05b-v6/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-05b-v6/model.safetensors:  14%|#3        |  136MB /  988MB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:26<00:00, 26.70s/it]


Unsloth: Merge process complete. Saved to `/content/daredevil467/hanoi-router-qwen25-05b-v6`
Pushed: https://huggingface.co/daredevil467/hanoi-router-qwen25-05b-v6


In [20]:
# ═══════════════════════ (OPTIONAL) Push GGUF Q4_K_M for Ollama production ═══════════════════════
# Run this cell only for the size you want to deploy via Ollama locally.
# GGUF export takes ~5-15 min depending on model size; safe to skip for ablation-only runs.

GGUF_HF_REPO = HF_REPO + '-gguf'
print(f'Pushing GGUF Q4_K_M to {GGUF_HF_REPO}...')
model.push_to_hub_gguf(
    GGUF_HF_REPO,
    tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN,
)
print(f'Pushed GGUF: https://huggingface.co/{GGUF_HF_REPO}')


Pushing GGUF Q4_K_M to daredevil467/hanoi-router-qwen25-05b-v6-gguf...
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/unsloth_gguf_h__rpquw`: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]


Successfully copied all 1 files from cache to `/tmp/unsloth_gguf_h__rpquw`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:04<00:00,  4.34s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_h__rpquw`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_h__rpquw_gguf/Qwen2.5-0.5B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_h__rpquw_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_h__rpquw_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /tmp/unsloth_gguf_h__rpquw_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /tmp/unsloth_gguf_h__rpquw_gguf/Modelfile
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading Qwen2.5-0.5B-Instruct.Q4_K_M.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0.5B-Instruct.Q4_K_M.gguf:  12%|#2        | 48.0MB /  398MB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/daredevil467/hanoi-router-qwen25-05b-v6-gguf
Unsloth: Cleaning up temporary files...
Pushed GGUF: https://huggingface.co/daredevil467/hanoi-router-qwen25-05b-v6-gguf
